# EDA — FIFA World Cup 2026 Player Performance

Análisis exploratorio del dataset procesado (salida del pipeline ETL) previo
al entrenamiento de los modelos de Machine Learning.

**Fuentes de datos:**
1. FIFA World Cup 2026 Player Performance Dataset (Kaggle)
2. País → Continente (Our World in Data / GitHub) — enriquecimiento


In [ ]:
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', 30)

df = pd.read_parquet('../data/processed/players_processed.parquet')
df.shape


In [ ]:
df.head()

## Calidad de datos

In [ ]:
print('Nulos totales:', df.isnull().sum().sum())
print('Duplicados:', df.duplicated().sum())
df.describe(include='all').T.head(20)


## Distribución de goles y asistencias

In [ ]:
fig = px.histogram(df, x='goal_contribution', title='Distribución de Goles + Asistencias por partido')
fig.show()


## Rendimiento por posición

In [ ]:
pos_summary = df.groupby('position')[['goals', 'assists', 'player_rating']].mean().round(2)
pos_summary


## Goles por continente (dimensión enriquecida con fuente externa)

In [ ]:
continent_goals = df.groupby('continent', as_index=False)['goals'].sum().sort_values('goals', ascending=False)
fig = px.bar(continent_goals, x='continent', y='goals', title='Goles totales por continente')
fig.show()


## Correlación entre variables numéricas clave

In [ ]:
cols = ['goals', 'assists', 'expected_goals_xg', 'expected_assists_xa', 'shots',
        'key_passes', 'player_rating', 'performance_score']
corr = df[cols].corr()
fig = px.imshow(corr, text_auto='.2f', title='Matriz de correlación', color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig.show()


## Conclusiones del EDA

- El dataset no presenta nulos ni duplicados tras el proceso de limpieza.
- La contribución de gol (goles + asistencias) está fuertemente desbalanceada
  (~9% de los registros con contribución > 0), lo que motivó el uso de
  `class_weight="balanced"` en el modelo de clasificación.
- `expected_goals_xg` y `expected_assists_xa` muestran correlación relevante
  con los goles y asistencias reales, confirmando su valor predictivo.
- El enriquecimiento con continente permite comparar rendimiento agregado
  por región geográfica en el dashboard.
